In [1]:
## conda env!!!

In [1]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES, RESULT_COLUMNS

#rewrite classes to use to only I class because it's only class I method
METHOD = 'mhcnuggets'
ORGANISMS_TO_USE = ['Human']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_A_02_01',
 'HLA_B_27_05',
 'HLA_A_03_01',
 'HLA_B_07_02',
 'HLA_DRB1_01_01',
 'HLA_DRB1_04_01',
 'HLA_DRB1_04_05']

In [3]:
# importing the predict module
from mhcnuggets.src.predict import predict
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import numpy as np
import pandas as pd

Using TensorFlow backend.


In [4]:
# Read datasets

DATA_PATH = "/home/denklewer/Data/"
INPUT_DATA_PATH = DATA_PATH + 'input/'
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'



In [6]:
from data_reading import read_genetech_data

per_allele_train_df, per_allele_val_df = read_genetech_data(DATA_PATH, CLASSES_TO_USE, ALLELES)

assert not 0 in [len(per_allele_val_df[allele_name]) for allele_name in ALLELES], 'One of dfs has length of 0, check data reading'


/home/denklewer/Tools/anacoda3/envs/mhcnuggets/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3361: DtypeWarning: Columns (6,7,8,9,14,21,22,23,24) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


In [7]:
df = per_allele_val_df[ALLELES[0]]
df.head()

,split,el,allele,peptide,n_flank,c_flank,mhc_a_1,mhc_a_2,mhc_b_1,mhc_b_2,...,gene_id,gene_symbol,n_samples,n_studies,Sample_ids,sample_names,Study_ids,study_names,negative_ensembl_id,binder
52052,test,1,HLA-A*02:01,AGIALNDNFVK,DTHSSIFDAK,LISWYDNEYG,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000105679,GAPDHS,1.0,1.0,1887,iedb_1034377_EL_A0201,5,iedb_consolidated,NaN,1
52053,test,1,HLA-A*02:01,AGLYRGLCAVRSR,GGAGPAVLQV,ALGLGLVSPA,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000123472,ATPAF1,1.0,1.0,1320,iedb_1029626_EL_A0201,5,iedb_consolidated,NaN,1
52054,test,1,HLA-A*02:01,AIDEKVQAV,APDIDKLKMA,HNLKMFWQST,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000100154,TTC28,1.0,1.0,1887,iedb_1034377_EL_A0201,5,iedb_consolidated,NaN,1
52055,test,1,HLA-A*02:01,AIFSAIIFL,VRHCFFTVIF,SFEAVSSLSK,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000005486,RHBDD2,1.0,1.0,677,Sarkizova_A0201,21,sarkizova_mhcI_2020,NaN,1
52056,test,1,HLA-A*02:01,AIIGGTFTV,PLYRFITTIC,AGILDSCIFT,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000113719,ERGIC1,2.0,1.0,"1356,1887","iedb_1019570_EL_A0201,iedb_1034377_EL_A0201",5,iedb_consolidated,NaN,1


In [8]:
def defineClass(allele_name):
    for organism_to_use in ORGANISMS_TO_USE:
        for class_to_use in CLASSES_TO_USE:
            if allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
                return class_to_use

In [13]:
def addColumnsAndFilter(per_allele_df):
    for allele_name in ALLELES:
        per_allele_df[allele_name]['source_sequence_name'] =  'seq' + per_allele_df[allele_name].index.astype(str).values
        per_allele_df[allele_name]['length'] = per_allele_df[allele_name]['peptide'].str.len()
        #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
        initial_len = len(per_allele_df[allele_name])
        if defineClass(allele_name) == 'I':
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name]['length'] <= 15]

        else:
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name]['length'] <= 30]
        result_len = len(per_allele_df[allele_name])
        print(f"Filtered out {initial_len - result_len} samples")
    
addColumnsAndFilter(per_allele_val_df)
addColumnsAndFilter(per_allele_train_df)

    



Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples


In [5]:
# different allele naming.
TARGET_ALLELES = dict()

## HARDCODE
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'HLA-DRB101:01'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'HLA-DRB104:01'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'HLA-DRB104:05'    
    else:
        target_allele_name = normalize_allele_name(allele_name).replace("*", "")
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA_A_02_01': 'HLA-A02:01',
 'HLA_B_27_05': 'HLA-B27:05',
 'HLA_A_03_01': 'HLA-A03:01',
 'HLA_B_07_02': 'HLA-B07:02',
 'HLA_DRB1_01_01': 'HLA-DRB101:01',
 'HLA_DRB1_04_01': 'HLA-DRB104:01',
 'HLA_DRB1_04_05': 'HLA-DRB104:05'}

## Training code

In [11]:
from mhcnuggets.src.train import train

In [28]:
def prepare_data(per_alelle_df):
    TEMPDIR = 'TMP/'
    for allele_name in ALLELES:
        df = per_alelle_df[allele_name]
        df['mhc'] = TARGET_ALLELES[allele_name]
        df.loc[df['binder'] == 1,'IC50(nM)'] = 100
        df.loc[df['binder'] == 0,'IC50(nM)'] = 50000
        df['measurement_type'] = 'qualitative'
        df['measurement_source'] = 'purified MHC/direct/fluorescence'
        df['original_allele'] = TARGET_ALLELES[allele_name]
        df[ ['mhc','peptide','IC50(nM)', 'measurement_type', 'measurement_source', 'original_allele']].to_csv(f"{TEMPDIR}mchnuggets_train_data_{allele_name}", index=False, header=True)    
        
prepare_data(per_allele_train_df)


def train_mhc_nuggets_models():
    TEMPDIR = 'TMP/'
    for allele_name in ALLELES:
        # ----------- Training with binding affinity data ----------------
        # training MHC class_I allele HLA-A02:01 using data present in the data file from scratch 
        class_to_use = defineClass(allele_name)
        target_allele = TARGET_ALLELES[allele_name]
        train(class_=class_to_use, 
              data=f'{TEMPDIR}mchnuggets_train_data_{allele_name}',
              mhc=target_allele,
              save_path=f'Models/mhcnuggets/{target_allele}_BA.h5', n_epoch=200)
        
        

train_mhc_nuggets_models()
        

    

    


# # predicting new line separated peptides present in the peptides_path file 
# # for MHC class_I allele HLA-A02:01. Since a MS_BA blend model is available,
# # this prediction will use the blended model (HLA-A02:01_BA_to_HLAp.h5)
# def predict_with_mhcnuggets(per_allele_df):
#     # save to temp dir
#     TEMPDIR = 'TMP/'
#     for allele_name in ALLELES:
#         per_alelle_df[allele_name]['peptide'].to_csv(f"{TEMPDIR}mchnuggets_train_data_{allele_name}", index=False, header=False)    
#     # predict
#     for allele_name in ALLELES:
#         class_to_use = defineClass(allele_name)
#         target_allele = TARGET_ALLELES[allele_name]
#         print(f"----------{target_allele}")
#         predict(class_=f'{class_to_use}',
#                 peptides_path=f'{TEMPDIR}mchnuggets_data_{allele_name}', 
#                 mhc=target_allele,
#                 output=f'{TEMPDIR}mchnuggets_predictions_{allele_name}')
#     # read predictions from temp
#     per_allele_predictions= {
#         allele_name :pd.read_csv(f"{TEMPDIR}mchnuggets_predictions_{allele_name}") for allele_name in ALLELES
#     }
#     for allele_name in per_allele_predictions.keys(): 
#         df_result = per_allele_predictions[allele_name]
#         # add binder label
        
#         per_allele_predictions[allele_name] = df_result.merge(per_allele_df[allele_name][[RESULT_COLUMNS['id'],RESULT_COLUMNS['label'], RESULT_COLUMNS['peptide']]], on=['peptide'])
#         # log-scale affinity
        
#         per_allele_predictions[allele_name]['score'] = 1 - np.log(per_allele_predictions[allele_name]['ic50'])/np.log(50000)
#     return per_allele_predictions
    

# ## create tempfiles for mhcnuggets



Training
MHC: HLA-A02:01
Data: TMP/mchnuggets_train_data_HLA_A_02_01
Model: lstm
Save path: Models/mhcnuggets/HLA_A_02_01_BA.h5
Transfer: None
MassSpec: False
Number of peptides skipped/total due to length 0 / 12400
epoch 0 / 200
Num pos: 6200.0000
Train AUC: 0.9242, F1: 0.8009, KTAU: 0.5999, PCC: 0.7475, PPV: 0.9186, PPVtop: 0.9186
epoch 1 / 200
Num pos: 6200.0000
Train AUC: 0.9312, F1: 0.7709, KTAU: 0.6098, PCC: 0.7687, PPV: 0.9376, PPVtop: 0.9376
epoch 2 / 200
Num pos: 6200.0000
Train AUC: 0.9341, F1: 0.6693, KTAU: 0.6140, PCC: 0.7759, PPV: 0.9616, PPVtop: 0.9616
epoch 3 / 200
Num pos: 6200.0000
Train AUC: 0.9384, F1: 0.6609, KTAU: 0.6200, PCC: 0.7857, PPV: 0.9662, PPVtop: 0.9662
epoch 4 / 200
Num pos: 6200.0000
Train AUC: 0.9438, F1: 0.8401, KTAU: 0.6276, PCC: 0.7960, PPV: 0.9437, PPVtop: 0.9437
epoch 5 / 200
Num pos: 6200.0000
Train AUC: 0.9469, F1: 0.7979, KTAU: 0.6320, PCC: 0.8068, PPV: 0.9631, PPVtop: 0.9631
epoch 6 / 200
Num pos: 6200.0000
Train AUC: 0.9517, F1: 0.8823, KTAU: 

In [22]:
per_allele_train_df[ALLELES[0]]

,split,el,allele,peptide,n_flank,c_flank,mhc_a_1,mhc_a_2,mhc_b_1,mhc_b_2,...,study_names,negative_ensembl_id,binder,source_sequence_name,length,mhc,IC50,measurement_type,measurement_source,original_allele
0,train,1,HLA-A*02:01,AAAIEVHEV,NDTFPTAMHI,LLPGLQKLHD,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,iedb_consolidated,NaN,1,seq0,9,HLA-A02:01,100.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01
1,train,1,HLA-A*02:01,AAALIIHHV,AYEDKQILED,KRQTGIQKED,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,iedb_consolidated,NaN,1,seq1,9,HLA-A02:01,100.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01
2,train,1,HLA-A*02:01,AAANIIRTL,DPELLTQERK,TQEDPGDNQI,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,iedb_consolidated,NaN,1,seq2,9,HLA-A02:01,100.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01
3,train,1,HLA-A*02:01,AAAPVVPAV,CSTTSTICTV,STSSAAGVAP,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,"sarkizova_mhcI_2020,iedb_consolidated",NaN,1,seq3,9,HLA-A02:01,100.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01
4,train,1,HLA-A*02:01,AAASLVPVV,QDLDDDVRAV,ESLVYLQTQK,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,iedb_consolidated,NaN,1,seq4,9,HLA-A02:01,100.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32221,train,0,HLA-A*02:01,EGDNEEEEYFVAQG,ELLLRSSQRE,QQ,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,NaN,1.0,0,seq32221,14,HLA-A02:01,50000.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01
32222,train,0,HLA-A*02:01,NKVITLRS,VLVQIDATEG,PLQIKNHFSI,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,NaN,1.0,0,seq32222,8,HLA-A02:01,50000.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01
32223,train,0,HLA-A*02:01,YLLATQHCAAV,HVLPAAQHSV,VSSLLGSPLP,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,NaN,1.0,0,seq32223,11,HLA-A02:01,50000.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01
32224,train,0,HLA-A*02:01,KLLDTMVDTF,VPPGTTISRV,LQKLVAAGSS,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,NaN,1.0,0,seq32224,10,HLA-A02:01,50000.0,qualitative,purified MHC/direct/fluorescence,HLA-A02:01


In [88]:
per_allele_predictions = predict_with_mhcnuggets(per_allele_val_df)

----------HLA-A02:01
Predicting for 12400 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-A02:01
BA_to_HLAp model found, predicting with BA_to_HLAp model...
----------HLA-B27:05
Predicting for 30420 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-B27:05
No BA_to_HLAp model found, predicting with BA model...
----------HLA-A03:01
Predicting for 3292 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-A03:01
BA_to_HLAp model found, predicting with BA_to_HLAp model...
----------HLA-B07:02
Predicting for 5940 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-B07:02
No BA_to_HLAp model found, predicting with BA model...
----------HLA-DRB101:01
Predicting for 49743 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-DRB101:01
No BA_to_HL

In [89]:
assert RESULT_COLUMNS['label'] in per_allele_predictions[ALLELES[0]].columns, f"label column must be in the result, this column is '{RESULT_COLUMNS['label']}'"
assert RESULT_COLUMNS['id'] in per_allele_predictions[ALLELES[0]].columns, f"id column must be in the result, this column is '{RESULT_COLUMNS['id']}'"
assert RESULT_COLUMNS['peptide'] in per_allele_predictions[ALLELES[0]].columns, f"peptide column must be in the result, this column is '{RESULT_COLUMNS['peptide']}'"

#### write to files

In [90]:
# for decoy_key in DECOY_PREFIXES:
for allele_name, total_df in per_allele_predictions.items():
    total_df.to_csv(f"{TARGET_PATH_TO_PREDICTIONS}{METHOD}_predictions_{allele_name}")

In [91]:
for allele_name in ALLELES:
    df = per_allele_predictions[allele_name]
    assert not df.isnull().values.any()
    assert not df.isna().values.any()
    assert np.isfinite(df['score']).all()
#     print(np.isnan().value_counts())
#     print(np.isnan(df['score']).value_counts())
#     print(np.isinf(df['score']).value_counts())